<a href="https://colab.research.google.com/github/NehalShahu/Gen_AI/blob/main/Prac_1_GenAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Representation using NLP Techniques

## Problem Statement:
A company wants to classify customer feedback. The task is to convert this feedback into machine-readable vectors using One-Hot Encoding and Bag of Words.

### 1. Data Preparation and Preprocessing

First, we'll start with some sample customer feedback and perform basic text preprocessing steps like lowercasing, removing punctuation, removing numbers, and removing stopwords. This cleaned data will then be used for encoding.

In [8]:
import re
import string
import nltk
from nltk.corpus import stopwords
import pandas as pd # Import pandas for displaying BoW results later

# Download stopwords if not already present
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')

sample_data = [
    'I absolutely LOVE this phone!!',
    'BatterY lasts 2 days ❤️❤️',
    'Worst Product Ever:(Money wasted!!',
    'Delivery was quick, but the packaging was damaged :( :(',
    'Excellent camera quality and amazing performance!!'
]

def remove_punctuation(text):
    return re.sub(r'[^A-z\s]', ' ', text) # Keep only English letters and spaces

def remove_numbers(text):
    return re.sub(r'\d+', '', text)

english_stopwords = set(stopwords.words('english'))

def remove_stopwords(text):
    return ' '.join(word for word in text.split() if word not in english_stopwords)

processed_data = []
for text in sample_data:
    text = text.lower()
    text = remove_punctuation(text)
    text = remove_numbers(text)
    text = remove_stopwords(text)
    processed_data.append(text.strip()) # strip any leading/trailing spaces

print("Original vs Processed Feedback:\n")
for original, processed in zip(sample_data, processed_data):
    print(f"Original : {original}")
    print(f"Processed: {processed}\n")

Original vs Processed Feedback:

Original : I absolutely LOVE this phone!!
Processed: absolutely love phone

Original : BatterY lasts 2 days ❤️❤️
Processed: battery lasts days

Original : Worst Product Ever:(Money wasted!!
Processed: worst product ever money wasted

Original : Delivery was quick, but the packaging was damaged :( :(
Processed: delivery quick packaging damaged

Original : Excellent camera quality and amazing performance!!
Processed: excellent camera quality amazing performance



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


### 2. Text Representation using One-Hot Encoding

One-Hot Encoding converts each unique word into a binary vector. Each vector has a length equal to the vocabulary size, with a '1' at the index corresponding to the word and '0's elsewhere. This allows machine learning models to process categorical data.

In [9]:
import numpy as np

# 1. Create a vocabulary of all unique words from the processed data
all_words = []
for text in processed_data:
    for word in text.split():
        all_words.append(word)

vocabulary = sorted(list(set(all_words)))
vocab_size = len(vocabulary)

print(f"Vocabulary: {vocabulary}")
print(f"Vocabulary size: {vocab_size}\n")

# 2. Implement One-Hot Encoding function
def one_hot_encode(text, vocabulary):
    encoded_vector = np.zeros(len(vocabulary))
    for word in text.split():
        try:
            idx = vocabulary.index(word)
            encoded_vector[idx] = 1
        except ValueError:
            # Handle words not in vocabulary (e.g., if using a pre-defined vocab)
            pass
    return encoded_vector

# Apply One-Hot Encoding to the processed sample data
one_hot_encoded_data = []
for text in processed_data:
    encoded_text = one_hot_encode(text, vocabulary)
    one_hot_encoded_data.append(encoded_text)

print("One-Hot Encoded Data (first 3 examples):\n")
for i in range(min(3, len(processed_data))):
    print(f"Original: '{sample_data[i]}'\nProcessed: '{processed_data[i]}'\nEncoded: {one_hot_encoded_data[i]}\n")

Vocabulary: ['absolutely', 'amazing', 'battery', 'camera', 'damaged', 'days', 'delivery', 'ever', 'excellent', 'lasts', 'love', 'money', 'packaging', 'performance', 'phone', 'product', 'quality', 'quick', 'wasted', 'worst']
Vocabulary size: 20

One-Hot Encoded Data (first 3 examples):

Original: 'I absolutely LOVE this phone!!'
Processed: 'absolutely love phone'
Encoded: [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 0.]

Original: 'BatterY lasts 2 days ❤️❤️'
Processed: 'battery lasts days'
Encoded: [0. 0. 1. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Original: 'Worst Product Ever:(Money wasted!!'
Processed: 'worst product ever money wasted'
Encoded: [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 1. 1.]



### 3. Text Representation using Bag of Words (BoW)

Bag of Words represents text as an unordered collection of words, focusing on word frequency. Each document is converted into a vector where each element counts how many times a word from the entire vocabulary appears in that document. The order of words is ignored.

In [10]:
from sklearn.feature_extraction.text import CountVectorizer

# 1. Initialize CountVectorizer (Bag of Words model)
# It will automatically create a vocabulary and count word occurrences.
vectorizer = CountVectorizer()

# 2. Fit and transform the processed data to create the BoW matrix
bow_matrix = vectorizer.fit_transform(processed_data)

# Get the feature names (words in the vocabulary)
vocab_bow = vectorizer.get_feature_names_out()

print(f"Bag of Words Vocabulary: {vocab_bow}")
print(f"Bag of Words Matrix Shape: {bow_matrix.shape}\n")

# Display the BoW matrix as a DataFrame for better readability
bow_df = pd.DataFrame(bow_matrix.toarray(), columns=vocab_bow)
print("Bag of Words Data (first 3 examples):\n")
for i in range(min(3, len(processed_data))):
    print(f"Original: '{sample_data[i]}'\nProcessed: '{processed_data[i]}'\nEncoded (counts):\n{bow_df.iloc[i].to_frame().T}\n")

Bag of Words Vocabulary: ['absolutely' 'amazing' 'battery' 'camera' 'damaged' 'days' 'delivery'
 'ever' 'excellent' 'lasts' 'love' 'money' 'packaging' 'performance'
 'phone' 'product' 'quality' 'quick' 'wasted' 'worst']
Bag of Words Matrix Shape: (5, 20)

Bag of Words Data (first 3 examples):

Original: 'I absolutely LOVE this phone!!'
Processed: 'absolutely love phone'
Encoded (counts):
   absolutely  amazing  battery  camera  damaged  days  delivery  ever  \
0           1        0        0       0        0     0         0     0   

   excellent  lasts  love  money  packaging  performance  phone  product  \
0          0      0     1      0          0            0      1        0   

   quality  quick  wasted  worst  
0        0      0       0      0  

Original: 'BatterY lasts 2 days ❤️❤️'
Processed: 'battery lasts days'
Encoded (counts):
   absolutely  amazing  battery  camera  damaged  days  delivery  ever  \
1           0        0        1       0        0     1         0     0   
